# Bakehouse — PII Masking & Data Security

**Dataset:** `samples.bakehouse` — sales_transactions, sales_customers, sales_franchises, media_customer_reviews  
**Difficulty:** Hard  
**Topics:** regexp_replace, substring, data quality, deduplication, anomaly detection, masking patterns


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

transactions = spark.read.table("samples.bakehouse.sales_transactions")
customers    = spark.read.table("samples.bakehouse.sales_customers")
franchises   = spark.read.table("samples.bakehouse.sales_franchises")
reviews      = spark.read.table("samples.bakehouse.media_customer_reviews")

print("Tables loaded.")


## Problem 1 — Advanced PII Masking

Build a full operational dataset joining `transactions` + `customers` + `franchises`. Apply masking:
- `first_name` and `last_name`: replace all chars with `*`
- `email`: keep first letter + `***` + `@domain`
- `phone`: keep last 3 digits, mask rest with `*`
- `cardNumber`: cast to string, keep last 4 digits, replace rest with `****-****-****-`

**Expected columns:** `transactionID`, `dateTime`, `franchiseName`, `firstName`, `lastName`, `emailAddress`, `phoneNumber`, `city`, `country`, `product`, `quantity`, `totalPrice`, `paymentMethod`, `maskedCard`


In [ ]:
# Problem 1 — write your solution here
#        F.concat(F.substring(email,1,1), F.lit('***'), F.lit('@'), domain) for email
#        F.concat(F.repeat('*',len-3), F.substring(phone,-3,3)) for phone
# Assign result to: result_1

result_1 = None  # replace this


In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
for expected in ['transactionid', 'datetime', 'franchisename', 'firstname', 'lastname',
                  'emailaddress', 'phonenumber', 'city', 'country', 'product',
                  'quantity', 'totalprice', 'paymentmethod', 'maskedcard']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_1.count()
assert cnt > 0, f"Got 0 rows"
sample = result_1.limit(1).collect()[0]
assert all(c == '*' for c in sample['firstName']), "firstName should be fully masked with *"
assert '@' in sample['emailAddress'], "emailAddress should still contain @"
assert sample['maskedCard'].startswith('****'), "maskedCard should start with ****"
print(f"Problem 1 passed ✓  ({cnt} rows)")


## Problem 2 — Email Domain Analysis

From `sales_customers` extract the domain from `email_address` using `split` or `regexp_extract`.  
Count customers per domain, find top 10. Verify domain column has no `@` symbols.

**Expected columns:** `email_domain`, `customer_count`


In [ ]:
# Problem 2 — write your solution here
# Assign result to: result_2

result_2 = None  # replace this


In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
for expected in ['email_domain', 'customer_count']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_2.count()
assert cnt > 0, f"Got 0 rows"
assert cnt <= 10, f"Expected top 10 domains, got {cnt}"
at_in_domain = result_2.filter(F.col('email_domain').contains('@')).count()
assert at_in_domain == 0, f"email_domain column has {at_in_domain} rows containing '@'"
assert result_2.filter(F.col('customer_count') <= 0).count() == 0, "customer_count must be > 0"
print(f"Problem 2 passed ✓  ({cnt} rows)")


## Problem 3 — Customer Deduplication

Find customers with the same `(first_name, last_name, country)` — potential duplicates.  
Use `groupBy` + `collect_list`. Filter where `duplicate_count > 1`.

**Expected columns:** `first_name`, `last_name`, `country`, `duplicate_count`, `customer_ids`


In [ ]:
# Problem 3 — write your solution here
#         F.collect_list('customerID').alias('customer_ids')).filter(F.col('duplicate_count') > 1)
# Assign result to: result_3

result_3 = None  # replace this


In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
for expected in ['first_name', 'last_name', 'country', 'duplicate_count', 'customer_ids']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_3.count()
assert cnt > 0, f"Got 0 rows — expected some duplicates"
assert result_3.filter(F.col('duplicate_count') <= 1).count() == 0, "Groups with count <= 1 should be filtered out"
id_type = result_3.schema['customer_ids'].dataType
assert isinstance(id_type, T.ArrayType), "customer_ids must be an ArrayType"
print(f"Problem 3 passed ✓  ({cnt} duplicate groups)")


## Problem 4 — Payment Card Anomaly Detection

In `sales_transactions`, find `cardNumber`s that appear for more than 3 distinct `customerID`s.  
Mask card as: cast to string, show last 4 digits preceded by `****`.

**Expected columns:** `masked_card`, `distinct_customers`, `transaction_count`


In [ ]:
# Problem 4 — write your solution here
#        masked = F.concat(F.lit('****'), last4)
#        groupBy cardNumber, filter countDistinct(customerID) > 3
# Assign result to: result_4

result_4 = None  # replace this


In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
for expected in ['masked_card', 'distinct_customers', 'transaction_count']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_4.count()
assert cnt >= 0, "count must be non-negative"
if cnt > 0:
    assert result_4.filter(F.col('distinct_customers') <= 3).count() == 0, "All results should have > 3 distinct customers"
    sample_mask = result_4.select('masked_card').first()['masked_card']
    assert sample_mask.startswith('****'), f"masked_card should start with '****', got: {sample_mask}"
print(f"Problem 4 passed ✓  ({cnt} suspicious cards)")


## Problem 5 — Transaction Velocity Anomaly

Find customers who made 3+ transactions within any 1-hour window.  
Use window functions partitioned by `customerID`, ordered by `dateTime`.  
Use `lag` to compute time difference between consecutive transactions.

**Expected columns:** `customerID`, `dateTime`, `transactions_in_last_hour`  
Filter where `transactions_in_last_hour >= 3`.


In [ ]:
# Problem 5 — write your solution here
#        transactions_in_last_hour = F.count('transactionID').over(w)
#        filter >= 3
# Assign result to: result_5

result_5 = None  # replace this


In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
for expected in ['customerid', 'datetime', 'transactions_in_last_hour']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_5.count()
assert cnt >= 0, "count must be non-negative"
if cnt > 0:
    assert result_5.filter(F.col('transactions_in_last_hour') < 3).count() == 0, "All rows must have >= 3 transactions in last hour"
print(f"Problem 5 passed ✓  ({cnt} high-velocity rows)")


## Problem 6 — Revenue Reconciliation Audit

Verify `totalPrice == unitPrice * quantity` for every transaction. Find and quantify discrepancies.

**Detail columns:** `transactionID`, `product`, `unitPrice`, `quantity`, `totalPrice`, `expected_price`, `discrepancy`

Then create a summary: `total_rows`, `matching_rows`, `discrepancy_rows`, `total_discrepancy_value`.


In [ ]:
# Problem 6 — write your solution here
#        result_6 = detail df with discrepancy != 0
# Assign detail result to: result_6

result_6 = None  # replace this


In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
for expected in ['transactionid', 'product', 'unitprice', 'quantity', 'totalprice', 'expected_price', 'discrepancy']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_6.count()
assert cnt >= 0, "count must be non-negative"
if cnt > 0:
    zero_disc = result_6.filter(F.col('discrepancy') == 0).count()
    assert zero_disc == 0, f"Found {zero_disc} rows with zero discrepancy — should be excluded"
print(f"Problem 6 passed ✓  ({cnt} discrepant transactions)")


## Problem 7 — Geographic Revenue Heatmap

Join `transactions` + `franchises`. Round `lat` and `lon` to 1 decimal (`lat_bucket`, `lon_bucket`).  
Per bucket: `total_revenue`, `transaction_count`, `unique_customers`.

**Expected columns:** `lat_bucket`, `lon_bucket`, `total_revenue`, `transaction_count`, `unique_customers`


In [ ]:
# Problem 7 — write your solution here
# Assign result to: result_7

result_7 = None  # replace this


In [ ]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
for expected in ['lat_bucket', 'lon_bucket', 'total_revenue', 'transaction_count', 'unique_customers']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_7.count()
assert cnt > 0, f"Got 0 rows"
assert result_7.filter(F.col('total_revenue') <= 0).count() == 0, "total_revenue must be > 0"
assert result_7.filter(F.col('transaction_count') <= 0).count() == 0, "transaction_count must be > 0"
assert result_7.filter(F.col('unique_customers') <= 0).count() == 0, "unique_customers must be > 0"
print(f"Problem 7 passed ✓  ({cnt} grid cells)")


## Problem 8 — Data Quality Report

For each of the 4 sales tables (transactions, customers, franchises, media_customer_reviews),  
compute: `row_count`, `column_count`, and `columns_with_nulls` (comma-joined string of column names  
that have any null values). Use a loop over a dict of dataset names.

**Expected columns:** `table_name`, `row_count`, `column_count`, `columns_with_nulls`


In [ ]:
# Problem 8 — write your solution here
#        for each col check df.filter(F.col(c).isNull()).count() > 0
#        build rows list, then spark.createDataFrame(rows, schema)
# Assign result to: result_8

result_8 = None  # replace this


In [ ]:
# ── Tests for Problem 8 ──────────────────────────────────────────
assert result_8 is not None, "result_8 is None"
assert hasattr(result_8, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_8.columns]
for expected in ['table_name', 'row_count', 'column_count', 'columns_with_nulls']:
    assert expected in cols, f"Missing column: {expected}"
cnt = result_8.count()
assert cnt == 4, f"Expected exactly 4 rows (one per table), got {cnt}"
assert result_8.filter(F.col('row_count') <= 0).count() == 0, "row_count must be > 0"
assert result_8.filter(F.col('column_count') <= 0).count() == 0, "column_count must be > 0"
tnames = {r['table_name'] for r in result_8.select('table_name').collect()}
assert len(tnames) == 4, f"Expected 4 distinct table names, got {len(tnames)}"
print(f"Problem 8 passed ✓  ({cnt} rows)")
